<a href="https://colab.research.google.com/github/sandipankar-data-engineer/data-engineering-scenarios-pandas/blob/sample-pandas-rm11062026/JSON_Normalize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dummy Data

In [ ]:
data = [
    {
        "id": "A",
        "name": "Alice",
        "address": {"city": "New York", "zip": "10001"},
        "grades": [{"math": 90, "physics": 85}, {"math": 88, "physics": 92}],
        "tpoint": [{"tp1":[10,20,30],"tp2":[40,50,60]},{"tp1":[11,21,31],"tp2":[41,51,61]}]

    },
    {
        "id": "B",
        "name": "Bob",
        "address": {"city": "San Francisco", "zip": "94102"},
        "grades": [{"math": 75, "physics": 80}]
    }
]

In [ ]:
complex_json = {
    "batch_id": "BATCH_001",
    "generated_at": "2026-02-15T10:30:00Z",
    "source": {
        "system": "CRM",
        "region": "APAC",
        "version": {
            "major": 2,
            "minor": 5
        }
    },
    "customers": [
        {
            "customer_id": "C001",
            "name": {
                "first": "Alice",
                "last": "Johnson"
            },
            "contact": {
                "emails": ["alice@gmail.com", "alice.work@company.com"],
                "phones": [
                    {"type": "mobile", "number": "111-222-3333"},
                    {"type": "home", "number": "444-555-6666"}
                ]
            },
            "orders": [
                {
                    "order_id": "O1001",
                    "amount": 250.75,
                    "items": [
                        {"product_id": "P01", "qty": 2, "price": 50},
                        {"product_id": "P02", "qty": 1, "price": 150}
                    ],
                    "payments": [
                        {"method": "card", "status": "completed"},
                        {"method": "voucher", "status": "applied"}
                    ]
                },
                {
                    "order_id": "O1002",
                    "amount": 100,
                    "items": [],
                    "payments": None
                }
            ],
            "tags": ["premium", "newsletter_subscriber"]
        },
        {
            "customer_id": "C002",
            "name": {
                "first": "Bob",
                "last": "Smith"
            },
            "contact": {
                "emails": [],
                "phones": [
                    {"type": "mobile", "number": "777-888-9999"}
                ]
            },
            "orders": [
                {
                    "order_id": "O2001",
                    "amount": 500,
                    "items": [
                        {"product_id": "P03", "qty": 5, "price": 100}
                    ],
                    "payments": [
                        {"method": "upi", "status": "pending"}
                    ]
                }
            ],
            "tags": None
        }
    ],
    "metadata": {
        "record_count": 2,
        "flags": {
            "test_data": False,
            "priority": "high"
        }
    }
}


In [ ]:
import pandas as pd
import numpy as np
import random as random

## Method 1

In [ ]:
df = pd.DataFrame(data)

# Expand address
address_df = pd.json_normalize(df["address"])
df_address_normalize = pd.concat([df.drop(columns=["address"]), address_df], axis=1)

# Explode grades
df_grades_explode = df_address_normalize.explode("grades").reset_index(drop=True)   # 🔥 IMPORTANT

# Expand grades dictionary
grades_df = pd.json_normalize(df_grades_explode["grades"])
df_result = pd.concat([df_grades_explode.drop(columns=["grades"]), grades_df], axis=1)

print(df_result)

  id   name                                             tpoint           city  \
0  A  Alice  [{'tp1': [10, 20, 30], 'tp2': [40, 50, 60]}, {...       New York   
1  A  Alice  [{'tp1': [10, 20, 30], 'tp2': [40, 50, 60]}, {...       New York   
2  B    Bob                                                NaN  San Francisco   

     zip  math  physics  
0  10001    90       85  
1  10001    88       92  
2  94102    75       80  


## Method 2

In [ ]:
# Normalize Json
df_normalize = pd.json_normalize(
    data,
    record_path="grades",
    meta=[
        "id",
        "name",
        ["address", "city"],
        ["address", "zip"]
    ]
)

# Rename columns
df_normalize.rename(columns={"address.city": "city","address.zip": "zip"}, inplace=True)

print(df_normalize)

   math  physics id   name           city    zip
0    90       85  A  Alice       New York  10001
1    88       92  A  Alice       New York  10001
2    75       80  B    Bob  San Francisco  94102


## Recursive Flatten

In [ ]:
def flatten_json(y, prefix=''):
    out = {}
    # Iterate through items of a dictionary
    for key, value in y.items():
        if isinstance(value, dict):
            # If dict, call flatten_json to inject further
            out.update(flatten_json(value, prefix + key + '_'))
        elif isinstance(value, list):
            # If list, iterate through the list
            for id,item in enumerate(value):
              # If list or dict, call flatten_json to inject further
              if isinstance(item,(list, dict)):
                out.update(flatten_json(item, prefix + key + '_' +str(id) + '_'))
              # If value, preserve the value in out variable
              else:
                out[prefix+key+'_'+str(id)] = item
        else:
            # If value, preserve the value in out variable
            out[prefix + key] = value

    return out

In [ ]:
def flatten_json_v2(y, prefix='', meta=None):
    if meta is None:
        meta = []
    out = {}
    counter = 1
    # Iterate through items of a dictionary
    for key, value in y.items():
        if isinstance(value, dict):
            # If dict, call flatten_json to inject further
            out.update(flatten_json_v2(value, prefix + key + '_',meta))
        elif isinstance(value, list):
            # If list, iterate through the list
            for id,item in enumerate(value):
              # If list or dict, call flatten_json to inject further
              if isinstance(item,(list, dict)):
                out.update(flatten_json_v2(item, prefix + key + '_' +str(id) + '_',meta))
              # If value, preserve the value in out variable
              else:
                col_name = prefix+key+'_'+str(id)
                while col_name in meta:
                  col_name = col_name + str(counter)
                  counter = counter + 1
                out[col_name] = item
                meta.append(col_name)
        else:
            # If value, preserve the value in out variable
            col_name = prefix + key
            while col_name in meta:
              col_name = col_name + str(counter)
              counter = counter + 1
            out[col_name] = value
            meta.append(col_name)

    return out

In [ ]:
bad_json_2 = {
    "a": {"b": 1},
    "a_b": 999
}

In [ ]:
flat = flatten_json_v2(complex_json['customers'][0])
sdf = pd.DataFrame([flat])
sdf

,customer_id,name_first,name_last,contact_emails_0,contact_emails_1,contact_phones_0_type,contact_phones_0_number,contact_phones_1_type,contact_phones_1_number,orders_0_order_id,...,orders_0_items_1_price,orders_0_payments_0_method,orders_0_payments_0_status,orders_0_payments_1_method,orders_0_payments_1_status,orders_1_order_id,orders_1_amount,orders_1_payments,tags_0,tags_1
0,C001,Alice,Johnson,alice@gmail.com,alice.work@company.com,mobile,111-222-3333,home,444-555-6666,O1001,...,150,card,completed,voucher,applied,O1002,100,None,premium,newsletter_subscriber


## Battleground: Complex JSON Data

### Normalize at customers

In [ ]:
df_json = pd.json_normalize(complex_json, record_path='customers', meta=['batch_id','generated_at',['source','system'],['source','region'],['source','version','major'],['source','version','minor'],['metadata','record_count'],['metadata','flags','test_data'],['metadata','flags','priority']])
df_json

,customer_id,orders,tags,name.first,name.last,contact.emails,contact.phones,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority
0,C001,"[{'order_id': 'O1001', 'amount': 250.75, 'item...","[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high
1,C002,"[{'order_id': 'O2001', 'amount': 500, 'items':...",None,Bob,Smith,[],"[{'type': 'mobile', 'number': '777-888-9999'}]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high


In [ ]:
df_json.columns

Index(['customer_id', 'orders', 'tags', 'name.first', 'name.last',
       'contact.emails', 'contact.phones', 'batch_id', 'generated_at',
       'source.system', 'source.region', 'source.version.major',
       'source.version.minor', 'metadata.record_count',
       'metadata.flags.test_data', 'metadata.flags.priority'],
      dtype='object')

### Exploding on contact.emails

In [ ]:
df_json_contact_emails = df_json.explode('contact.emails').reset_index()
df_json_contact_emails

,index,customer_id,orders,tags,name.first,name.last,contact.emails,contact.phones,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority
0,0,C001,"[{'order_id': 'O1001', 'amount': 250.75, 'item...","[premium, newsletter_subscriber]",Alice,Johnson,alice@gmail.com,"[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high
1,0,C001,"[{'order_id': 'O1001', 'amount': 250.75, 'item...","[premium, newsletter_subscriber]",Alice,Johnson,alice.work@company.com,"[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high
2,1,C002,"[{'order_id': 'O2001', 'amount': 500, 'items':...",None,Bob,Smith,NaN,"[{'type': 'mobile', 'number': '777-888-9999'}]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high


### Exploding on contact.phones

In [ ]:
df_json_contact_phones = df_json.explode('contact.phones').reset_index()
df_json_contact_phones_normalize = pd.json_normalize(df_json_contact_phones['contact.phones'])
df_json_contact_phones = pd.concat([df_json_contact_phones.drop(columns=['contact.phones']), df_json_contact_phones_normalize], axis=1)
df_json_contact_phones

,index,customer_id,orders,tags,name.first,name.last,contact.emails,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority,type,number
0,0,C001,"[{'order_id': 'O1001', 'amount': 250.75, 'item...","[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,mobile,111-222-3333
1,0,C001,"[{'order_id': 'O1001', 'amount': 250.75, 'item...","[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,home,444-555-6666
2,1,C002,"[{'order_id': 'O2001', 'amount': 500, 'items':...",None,Bob,Smith,[],BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,mobile,777-888-9999


### Exploding on orders

In [ ]:
df_json_contact_orders = df_json.explode('orders',ignore_index=True)
df_json_contact_orders_normalize = pd.json_normalize(df_json_contact_orders['orders'])
df_json_contact_orders = pd.concat([df_json_contact_orders.drop(columns=['orders']), df_json_contact_orders_normalize],axis=1)
df_json_contact_orders

,customer_id,tags,name.first,name.last,contact.emails,contact.phones,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority,order_id,amount,items,payments
0,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O1001,250.75,"[{'product_id': 'P01', 'qty': 2, 'price': 50},...","[{'method': 'card', 'status': 'completed'}, {'..."
1,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O1002,100.00,[],None
2,C002,None,Bob,Smith,[],"[{'type': 'mobile', 'number': '777-888-9999'}]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O2001,500.00,"[{'product_id': 'P03', 'qty': 5, 'price': 100}]","[{'method': 'upi', 'status': 'pending'}]"


#### Exploding on items

In [ ]:
df_json_contact_orders_items = df_json_contact_orders.explode('items',ignore_index=True)
df_json_contact_orders_items_normalize = pd.json_normalize(df_json_contact_orders_items['items'])
df_json_contact_orders_items = pd.concat([df_json_contact_orders_items.drop(columns='items'),df_json_contact_orders_items_normalize],axis=1)
df_json_contact_orders_items

,customer_id,tags,name.first,name.last,contact.emails,contact.phones,batch_id,generated_at,source.system,source.region,...,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority,order_id,amount,payments,product_id,qty,price
0,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,...,5,2,False,high,O1001,250.75,"[{'method': 'card', 'status': 'completed'}, {'...",P01,2.0,50.0
1,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,...,5,2,False,high,O1001,250.75,"[{'method': 'card', 'status': 'completed'}, {'...",P02,1.0,150.0
2,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,...,5,2,False,high,O1002,100.00,None,NaN,NaN,NaN
3,C002,None,Bob,Smith,[],"[{'type': 'mobile', 'number': '777-888-9999'}]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,...,5,2,False,high,O2001,500.00,"[{'method': 'upi', 'status': 'pending'}]",P03,5.0,100.0


#### Exploading on payments

In [ ]:
df_json_contact_orders_payments = df_json_contact_orders.explode('payments',ignore_index=True)
df_json_contact_orders_payments_normalize = pd.json_normalize(df_json_contact_orders_payments['payments'])
df_json_contact_orders_payments = pd.concat([df_json_contact_orders_payments.drop(columns='payments'),df_json_contact_orders_payments_normalize],axis=1)
df_json_contact_orders_payments

,customer_id,tags,name.first,name.last,contact.emails,contact.phones,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,metadata.flags.test_data,metadata.flags.priority,order_id,amount,items,method,status
0,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O1001,250.75,"[{'product_id': 'P01', 'qty': 2, 'price': 50},...",card,completed
1,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O1001,250.75,"[{'product_id': 'P01', 'qty': 2, 'price': 50},...",voucher,applied
2,C001,"[premium, newsletter_subscriber]",Alice,Johnson,"[alice@gmail.com, alice.work@company.com]","[{'type': 'mobile', 'number': '111-222-3333'},...",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O1002,100.00,[],NaN,NaN
3,C002,None,Bob,Smith,[],"[{'type': 'mobile', 'number': '777-888-9999'}]",BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,False,high,O2001,500.00,"[{'product_id': 'P03', 'qty': 5, 'price': 100}]",upi,pending


### Merge them into a single table

In [ ]:
# df_json - Customer
# df_json_contact_emails - Customer Email
# df_json_contact_phones - Customer Phone
# df_json_contact_orders - Orders
# df_json_contact_orders_items - Items
# df_json_contact_orders_payments - Payments

df_json_master_table = pd.merge(df_json.drop(columns='contact.emails'), df_json_contact_emails[['customer_id','contact.emails']], on='customer_id', how='left')
df_json_master_table = df_json_master_table.drop(columns=['tags'])
df_json_master_table = pd.merge(df_json_master_table.drop(columns='contact.phones'),df_json_contact_phones[['customer_id','type','number']], on='customer_id', how='left')
df_json_master_table = pd.merge(df_json_master_table.drop(columns='orders'),df_json_contact_orders[['customer_id','order_id','amount']], on='customer_id', how='left')
df_json_master_table = pd.merge(df_json_master_table,df_json_contact_orders_items[['customer_id','order_id','product_id','qty','price']], on=['customer_id','order_id'], how='left')
df_json_master_table = pd.merge(df_json_master_table,df_json_contact_orders_payments[['customer_id','order_id','method','status']], on=['customer_id','order_id'], how='left')
df_json_master_table

,customer_id,name.first,name.last,batch_id,generated_at,source.system,source.region,source.version.major,source.version.minor,metadata.record_count,...,contact.emails,type,number,order_id,amount,product_id,qty,price,method,status
0,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,mobile,111-222-3333,O1001,250.75,P01,2.0,50.0,card,completed
1,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,mobile,111-222-3333,O1001,250.75,P01,2.0,50.0,voucher,applied
2,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,mobile,111-222-3333,O1001,250.75,P02,1.0,150.0,card,completed
3,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,mobile,111-222-3333,O1001,250.75,P02,1.0,150.0,voucher,applied
4,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,mobile,111-222-3333,O1002,100.00,NaN,NaN,NaN,NaN,NaN
5,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,home,444-555-6666,O1001,250.75,P01,2.0,50.0,card,completed
6,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,home,444-555-6666,O1001,250.75,P01,2.0,50.0,voucher,applied
7,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,home,444-555-6666,O1001,250.75,P02,1.0,150.0,card,completed
8,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,home,444-555-6666,O1001,250.75,P02,1.0,150.0,voucher,applied
9,C001,Alice,Johnson,BATCH_001,2026-02-15T10:30:00Z,CRM,APAC,2,5,2,...,alice@gmail.com,home,444-555-6666,O1002,100.00,NaN,NaN,NaN,NaN,NaN
